In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import os
import random
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ====================== 随机种子 ======================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 数据加载 ======================
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

# ====================== 回归目标分数定义 ======================
def create_regression_targets(single_labels):
    batch_size = len(single_labels)
    dep_target = torch.zeros(batch_size, dtype=torch.float32)
    sui_target = torch.zeros(batch_size, dtype=torch.float32)
    
    for i, lab in enumerate(single_labels):
        if lab == 2:   # Normal
            dep_target[i] = 0.05
            sui_target[i] = 0.02
        elif lab == 0: # Anxiety
            dep_target[i] = 0.40
            sui_target[i] = 0.18
        elif lab == 1: # Depression
            dep_target[i] = 0.75
            sui_target[i] = 0.48
        elif lab == 3: # Suicidal
            dep_target[i] = 0.88
            sui_target[i] = 0.93
    return dep_target, sui_target

# ====================== Dataset ======================
class RiskRegressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        single_label = self.labels[idx]
        dep_target, sui_target = create_regression_targets([single_label])
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "dep_target": dep_target.squeeze(0),
            "sui_target": sui_target.squeeze(0),
            "single_label": torch.tensor(single_label, dtype=torch.long),
            "text": self.texts[idx]
        }
        return item

# ====================== 基础双回归模型 ======================
class DualRegressionModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        
        self.dep_head = nn.Linear(hidden, 1)
        self.sui_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask, dep_target=None, sui_target=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        
        dep_logit = self.dep_head(cls)
        sui_logit = self.sui_head(cls)
        
        dep_score = torch.sigmoid(dep_logit).squeeze(-1)
        sui_score = torch.sigmoid(sui_logit).squeeze(-1)
        
        if dep_target is not None:
            loss_dep = F.mse_loss(dep_score, dep_target)
            loss_sui = F.mse_loss(sui_score, sui_target)
            total_loss = loss_dep + 1.2 * loss_sui
            
            return {
                "loss": total_loss,
                "dep_score": dep_score,
                "sui_score": sui_score
            }
        
        return {
            "dep_score": dep_score,
            "sui_score": sui_score
        }

# ====================== Concordance Correlation Coefficient (CCC) ======================
def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

# ====================== 训练函数 ======================
def train_dual_regression():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    train_dataset = RiskRegressionDataset(train_texts, train_labels, tokenizer)
    val_dataset   = RiskRegressionDataset(val_texts,   val_labels,   tokenizer)
    test_dataset  = RiskRegressionDataset(test_texts,  test_labels,  tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
    
    model = DualRegressionModel(MODEL_PATH).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=1.8e-5)
    
    best_val_loss = float('inf')
    best_state = None
    
    for epoch in range(6):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].to(DEVICE)
            sui_target     = batch["sui_target"].to(DEVICE)
            
            outputs = model(input_ids, attention_mask, dep_target, sui_target)
            loss = outputs["loss"]
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Avg Train Loss (MSE): {avg_train_loss:.4f}")
        
        # Validation
        model.eval()
        val_dep_preds = []
        val_sui_preds = []
        val_dep_targets = []
        val_sui_targets = []
        val_labels_list = []
        
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                dep_target     = batch["dep_target"].to(DEVICE)
                sui_target     = batch["sui_target"].to(DEVICE)
                single_labels  = batch["single_label"].numpy()
                
                outputs = model(input_ids, attention_mask, dep_target, sui_target)
                val_loss += outputs["loss"].item()
                
                val_dep_preds.extend(outputs["dep_score"].cpu().numpy())
                val_sui_preds.extend(outputs["sui_score"].cpu().numpy())
                val_dep_targets.extend(dep_target.cpu().numpy())
                val_sui_targets.extend(sui_target.cpu().numpy())
                val_labels_list.extend(single_labels)
        
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1} Avg Val Loss (MSE): {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            torch.save(best_state, "best_dual_regression.pt")
            print(f"Best model saved (Val MSE: {avg_val_loss:.4f})")
    
    # ====================== Final Test ======================
    model.load_state_dict(torch.load("best_dual_regression.pt"))
    model.eval()
    
    test_dep_preds = []
    test_sui_preds = []
    test_dep_targets = []
    test_sui_targets = []
    test_labels_list = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dep_target     = batch["dep_target"].cpu().numpy()
            sui_target     = batch["sui_target"].cpu().numpy()
            single_labels  = batch["single_label"].numpy()
            
            outputs = model(input_ids, attention_mask)
            
            test_dep_preds.extend(outputs["dep_score"].cpu().numpy())
            test_sui_preds.extend(outputs["sui_score"].cpu().numpy())
            test_dep_targets.extend(dep_target)
            test_sui_targets.extend(sui_target)
            test_labels_list.extend(single_labels)
    
    # ====================== 回归指标 ======================
    dep_spear, _ = spearmanr(test_dep_targets, test_dep_preds)
    sui_spear, _ = spearmanr(test_sui_targets, test_sui_preds)
    
    dep_ccc = concordance_correlation_coefficient(test_dep_targets, test_dep_preds)
    sui_ccc = concordance_correlation_coefficient(test_sui_targets, test_sui_preds)
    
    dep_mae = np.mean(np.abs(np.array(test_dep_targets) - np.array(test_dep_preds)))
    sui_mae = np.mean(np.abs(np.array(test_sui_targets) - np.array(test_sui_preds)))
    
    print("\n=== Regression Metrics (Test Set) ===")
    print(f"Depression - Spearman: {dep_spear:.4f} | CCC: {dep_ccc:.4f} | MAE: {dep_mae:.4f}")
    print(f"Suicidal Risk - Spearman: {sui_spear:.4f} | CCC: {sui_ccc:.4f} | MAE: {sui_mae:.4f}")
    
    # ====================== 高风险检测（Suicidal） ======================
    high_risk_true = (np.array(test_labels_list) == 3).astype(int)
    threshold = 0.7
    high_risk_pred = (np.array(test_sui_preds) >= threshold).astype(int)
    
    auc = roc_auc_score(high_risk_true, test_sui_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(high_risk_true, high_risk_pred, average='binary')
    
    print(f"\n=== High-risk Detection (threshold = {threshold}) ===")
    print(f"AUC: {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1: {f1:.4f}")
    
    # ====================== 2D Risk Space 可视化 ======================
    plt.figure(figsize=(10, 8))
    colors = ['gray', 'blue', 'orange', 'red']  # Normal, Anxiety, Depression, Suicidal
    class_names = label_encoder.classes_
    severity_order = [2, 0, 1, 3]  # Normal(2), Anxiety(0), Depression(1), Suicidal(3)
    
    for idx, orig_lbl in enumerate(severity_order):
        name = class_names[orig_lbl]
        mask = np.array(test_labels_list) == orig_lbl
        plt.scatter(np.array(test_dep_preds)[mask], 
                    np.array(test_sui_preds)[mask], 
                    c=colors[idx], label=name, alpha=0.7, s=25)
    
    plt.axvline(x=0.68, color='orange', linestyle='--', alpha=0.7, label='Dep thresh (example)')
    plt.axhline(y=0.7, color='red', linestyle='--', alpha=0.7, label='Sui high-risk thresh')
    
    plt.xlabel('Predicted Depression Severity')
    plt.ylabel('Predicted Suicidal Risk')
    #plt.title('2D Risk Space - Test Set (Base Dual-Regression)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('2d_risk_space_dual_regression.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("\n2D risk scatter plot 已保存为 2d_risk_space_dual_regression.png")
    
    # ====================== 分数分布 Violin Plot ======================
    severity_class_names = ['Normal', 'Anxiety', 'Depression', 'Suicidal']
    severity_mapping = {2: 'Normal', 0: 'Anxiety', 1: 'Depression', 3: 'Suicidal'}
    
    df_violin = pd.DataFrame({
        'Dep Score': test_dep_preds,
        'Sui Score': test_sui_preds,
        'True Class': [severity_mapping[l] for l in test_labels_list]
    })
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(data=df_violin.melt(id_vars='True Class', value_vars=['Dep Score', 'Sui Score']),
                   x='True Class', y='value', hue='variable', split=True, inner='quartile',
                   order=severity_class_names)  # 强制按严重度顺序排序
    plt.title('Score Distribution by True Class')
    plt.ylabel('Predicted Score (0-1)')
    plt.tight_layout()
    plt.savefig('score_distribution_violin.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("分数分布 violin plot 已保存为 score_distribution_violin.png")
    
    print("\nBase Dual-Regression 实验完成！")
    print("请查看生成的图形和打印的指标。")

if __name__ == "__main__":
    train_dual_regression()


Using device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [07:41<00:00,  4.70it/s]


Epoch 1 Avg Train Loss (MSE): 0.0678
Epoch 1 Avg Val Loss (MSE): 0.0437
Best model saved (Val MSE: 0.0437)


Epoch 2 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [08:37<00:00,  4.19it/s]


...

Epoch 4 Avg Train Loss (MSE): 0.0297
Epoch 4 Avg Val Loss (MSE): 0.0389
Best model saved (Val MSE: 0.0389)


Epoch 5 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [09:05<00:00,  3.98it/s]


Epoch 5 Avg Train Loss (MSE): 0.0256
Epoch 5 Avg Val Loss (MSE): 0.0381
Best model saved (Val MSE: 0.0381)


Epoch 6 Train: 100%|█████████████████████████████████████████████████████████████| 2171/2171 [08:44<00:00,  4.14it/s]


Epoch 6 Avg Train Loss (MSE): 0.0220
Epoch 6 Avg Val Loss (MSE): 0.0440

=== Regression Metrics (Test Set) ===
Depression - Spearman: 0.9091 | CCC: 0.9484 | MAE: 0.0490
Suicidal Risk - Spearman: 0.9097 | CCC: 0.9083 | MAE: 0.0790

=== High-risk Detection (threshold = 0.7) ===
AUC: 0.9533
Precision: 0.7768
Recall: 0.8026
F1: 0.7895

2D risk scatter plot 已保存为 2d_risk_space_dual_regression.png
分数分布 violin plot 已保存为 score_distribution_violin.png

Base Dual-Regression 实验完成！
请查看生成的图形和打印的指标。
